# 05 Batch Normalization — CIFAR-100 with BN

## Batch Normalization

**Batch Normalization (BN)** normalizes the activations of each layer across the mini-batch, keeping them centered near zero with unit variance. This stabilizes training, allows higher learning rates, and acts as a mild regularizer.

```
output = (x - mean) / sqrt(var + ε) * γ + β
```

where `mean` and `var` are computed over the current mini-batch, and `γ`, `β` are learnable scale/shift parameters.

BN is applied **after each convolution and fully-connected layer, before ReLU**:

```
Conv → BN → ReLU → Pool
FC   → BN → ReLU
```

Two variants are used depending on the layer type:

| Class | Used after | Input shape |
|-------|-----------|-------------|
| `BatchNorm2d(C)` | Conv layer | `(B, C, H, W)` — normalizes over B, H, W per channel |
| `BatchNorm1d(C)` | FC layer | `(B, C)` — normalizes over B per feature |

> 📌 Like Dropout, BN behaves differently in training vs evaluation mode. In `net.eval()`, it uses **running statistics** (accumulated during training) instead of the current batch.

| Mode | `net.train()` | `net.eval()` |
|------|:---:|:---:|
| Uses batch statistics | ✅ | ❌ |
| Uses running statistics | ❌ | ✅ |

📄 [nn.BatchNorm2d docs](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH="/content/drive/MyDrive/Practical-ML/cifar100"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls *.py

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile 05BN.py
# Adapted for CIFAR-100 with Batch Normalization

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
import torch.optim.lr_scheduler as sch
import torch.backends.cudnn as cudnn

import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

batch_size = 32
epochs = 10

# training data preparation
transform = transforms.Compose(
    [transforms.RandomHorizontalFlip(),          # flip left<->right with 50% probability
     transforms.RandomCrop(32, padding=4),       # pad 4px each side, then crop back to 32x32
     transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
# Note: for a fair evaluation, test transform should ideally use only ToTensor+Normalize.
# Here we reuse the same transform variable for simplicity.
testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

# CIFAR-100 has 100 classes (20 superclasses x 5 subclasses each)

# network definition
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        self.conv1 = nn.Conv2d(3, 16, 3, padding=1, padding_mode='replicate')
        self.conv1_bn = nn.BatchNorm2d(16)    # normalize over (B,H,W) per channel
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1, padding_mode='zeros')
        self.conv2_bn = nn.BatchNorm2d(32)    # normalize over (B,H,W) per channel

        self.pool = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc1_bn = nn.BatchNorm1d(128)     # normalize over B per feature
        self.fc2 = nn.Linear(128, 64)
        self.fc2_bn = nn.BatchNorm1d(64)      # normalize over B per feature
        self.fc3 = nn.Linear(64, 100)  # 100 classes

        self.dropout = nn.Dropout(0.5)        # randomly zero 50% of neurons during training

    def forward(self, x):
        x = self.conv1(x) # (B,3,32,32) -> (B,16,32,32)
        x = self.conv1_bn(x)  # batch normalize
        x = F.relu(x)
        x = self.pool(x) # (B,16,32,32) -> (B,16,16,16)

        x = self.conv2(x) # (B,16,16,16) -> (B,32,16,16)
        x = self.conv2_bn(x)  # batch normalize
        x = F.relu(x)
        x = self.pool(x) # (B,32,16,16) -> (B,32,8,8)

        x = self.flatten(x) # (B,32,8,8) -> (B,32*8*8)

        x = self.fc1(x) # (B,32*8*8) -> (B,128)
        x = self.fc1_bn(x)    # batch normalize
        x = F.relu(x)

        x = self.fc2(x) # (B,128) -> (B,64)
        x = self.fc2_bn(x)    # batch normalize
        x = F.relu(x)

        x = self.dropout(x)   # apply dropout before the final layer
        x = self.fc3(x) # (B,64) -> (B,100)

        return x

net = Net()
net = net.to(device)
if device == 'cuda':
    cudnn.benchmark = True
    print('Run with GPU')
else:
    print('Run with CPU')

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
scheduler = sch.StepLR(optimizer, step_size=1, gamma=0.9)

net.train()
t0 = time.perf_counter()
for epoch in range(epochs):

    running_loss = 0.0
    for i, data in enumerate(trainloader):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if i % 100 == 99:
            t1 = time.perf_counter()
            print('[%d, %5d, %.2e] loss: %.3f, time: %.3f' %
                  (epoch + 1, i + 1, optimizer.param_groups[0]['lr'], running_loss / 2000, t1-t0))
            running_loss = 0.0
            t0 = t1
    scheduler.step() # invoke learning rate scheduler after each epoch

print('Finished Training')

net.eval()
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 10000 test images: %.3f %%' % (100 * correct / total))

In [ ]:
# Execute a Python file
%cd "{PROJECT_PATH}"
!python 05BN.py

💾 **Don't forget to save this notebook!**

To keep your work, go to File → Save a copy in Drive before closing.

[![Return to GitHub](https://img.shields.io/badge/GitHub-Return-black?logo=github)](https://github.com/mastnk/practical-ml)